# CCW Torque VS Rotator angle analysis

Ticket: SITCOM-1853

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.time import Time
from pathlib import Path
from datetime import datetime

from lsst.summit.utils.tmaUtils import (
    TMAEventMaker,
    TMAState,
)
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

## Create Event Maker

We want to create a single instance of the `TMAEventMaker` object.  
Each instance might be quite heavy. 

In [ ]:
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)

event_maker = TMAEventMaker()
efd_client = makeEfdClient()

# Data Analysis

## Single Day

We are looking for data during one day.  
This comes with the first filter below.  
There are situations where `actualTorquePercentage` reports only `None` values. We need to filter these out. 

In [ ]:
day_obs = 20241207
all_events = event_maker.getEvents(day_obs)
slew_events = [e for e in all_events if e.type == TMAState.SLEWING]

In [ ]:
evt = all_events[0]
print(evt)

## Data from the CCW

In [ ]:
ccw = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTMount.cameraCableWrap",
    columns=["actualTorquePercentage0", "actualTorquePercentage1", "actualPosition", "actualPositionTimestamp"],
    event=evt,
)

In [ ]:
ccw

## Data from the rotator

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTPtg.mountPosition",
    columns=["rotatorActualPosition"],
    event=evt,
)

In [ ]:
rot

In [ ]:
day_obs = 20241207
all_events = event_maker.getEvents(day_obs)
slew_events = [e for e in all_events if e.type == TMAState.SLEWING]

day_obs_df = pd.DataFrame(
    columns=[
        "seq_num",
        "torque_min_0",
        "torque_avg_0",
        "torque_max_0",
        "torque_min_1",
        "torque_avg_1",
        "torque_max_1",
        "rotator_actual_position",
    ]
)

for evt in slew_events:
    df = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTMount.cameraCableWrap",
        columns=[
            "actualPosition",
            "actualTorquePercentage0",
            "actualTorquePercentage1",
            "actualVelocity",
        ],
        event=evt,
        warn=False,
    )

    if len(df) == 0:
        print(f"dayObs = {evt.dayObs}, seqNum = {evt.seqNum} - Empty dataframe")
        continue

    # Fetch rotator position
    rot = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTPtg.mountPosition",
        columns=["rotatorActualPosition"],
        event=evt,
    )

    # Extract rotator position value (if available)
    rot_value = rot["rotatorActualPosition"].iloc[0] if not rot.empty else None

    # Compute the statistics
    stats = {
        f"torque_{stat}_{i}": df[f"actualTorquePercentage{i}"].agg(stat)
        for i in [0, 1]
        for stat in ["min", "mean", "max"]
    }
    stats["seq_num"] = evt.seqNum
    stats["max_abs_torque_0"] = df["actualTorquePercentage0"].abs().max()
    stats["max_abs_torque_1"] = df["actualTorquePercentage1"].abs().max()
    stats["rotator_actual_position"] = rot_value

    # Append the stats to the DataFrame using concat
    new_row = pd.DataFrame([stats])
    if day_obs_df.empty:
        day_obs_df = new_row  # Directly assign if day_obs_df is empty
    else:
        day_obs_df = pd.concat([day_obs_df, new_row], ignore_index=True)


In [ ]:
day_obs_df

In [ ]:
print(day_obs_df.columns)


Next plot will show the scatter of the CCW torques depending on rotator angle for one day of ComCam on sky

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot for Mean Torque 0 and Torque 1
sns.scatterplot(
    x=day_obs_df["rotator_actual_position"], 
    y=day_obs_df["torque_mean_0"],  
    label="Torque 0 (Mean)", 
    color="blue",
    alpha=0.7,      s=10  
)

sns.scatterplot(
    x=day_obs_df["rotator_actual_position"], 
    y=day_obs_df["torque_mean_1"],  
    label="Torque 1 (Mean)", 
    color="red",
    alpha=0.7,
    s=10
)

# Scatter plot for Min and Max Torque 0
sns.scatterplot(
    x=day_obs_df["rotator_actual_position"],
    y=day_obs_df["torque_min_0"],
    color="blue",
    alpha=0.3,
    s=5,
    label="Torque 0 (Min)"
)

sns.scatterplot(
    x=day_obs_df["rotator_actual_position"],
    y=day_obs_df["torque_max_0"],
    color="blue",
    alpha=0.3,
    s=5,
    label="Torque 0 (Max)"
)

# Scatter plot for Min and Max Torque 1
sns.scatterplot(
    x=day_obs_df["rotator_actual_position"],
    y=day_obs_df["torque_min_1"],
    color="red",
    alpha=0.3,
    s=5,
    label="Torque 1 (Min)"
)

sns.scatterplot(
    x=day_obs_df["rotator_actual_position"],
    y=day_obs_df["torque_max_1"],
    color="red",
    alpha=0.3,
    s=5,
    label="Torque 1 (Max)"
)

plt.xlabel("Rotator Actual Position (degrees)")
plt.ylabel("Torque (%)")
plt.title("Torque vs. Rotator Position (Scatter Plot with Min/Max Values)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()




# Mean Torque VS rotator angle for one day

In this plot we can't see any correlation between the CCW torque and rotator angle for one day of observation

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot for Torque 0
sns.scatterplot(
    x=day_obs_df["rotator_actual_position"], 
    y=day_obs_df["torque_mean_0"],  
    label="Torque 0 (Mean)", 
    color="blue", 
    s=50  
)

# Scatter plot for Torque 1
sns.scatterplot(
    x=day_obs_df["rotator_actual_position"], 
    y=day_obs_df["torque_mean_1"],  
    label="Torque 1 (Mean)", 
    color="red", 
    s=50  
)

plt.xlabel("Rotator Actual Position (degrees)")
plt.ylabel("Torque (%)")
plt.title("Torque vs. Rotator Position")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()


In [ ]:
print(day_obs_df["torque_mean_1"].min())  # Min value
print(day_obs_df["torque_mean_1"].max())  # Max value
print(day_obs_df["torque_mean_1"].describe())  # Summary statistics


## let's bin the data on every 5 degrees in rotator angle

In [ ]:
# Bin the data by 5 degrees
day_obs_df['rotator_angle_bin'] = (day_obs_df['rotator_actual_position'] // 5) * 5

stats = day_obs_df.groupby('rotator_angle_bin')['torque_mean_1'].describe()

print(stats)


## Violine plot 

1. A wide portion of the violin at, say, -5° would mean that torque values are frequently around that angle.
   
2. A narrower portion at 20° suggests that torque values at that angle are less frequent or less consistent.

3. The median line within the violin gives a good sense of the central tendency (where most of the data are centered).

4. and, if there are any thin tails (i.e., the violin is narrow at some values), it suggests that data points at those values are sparse.

Again, the results are not showing any significant dependence between the torques and the rotator angles. 


In [ ]:
# Bin the data by 5 degrees
day_obs_df['rotator_angle_bin'] = (day_obs_df['rotator_actual_position'] // 5) * 5

plt.figure(figsize=(12, 6))

sns.violinplot(
    x='rotator_angle_bin', 
    y='torque_mean_1', 
    data=day_obs_df, 
    inner="points", 
    color="red",  
)

plt.xlabel("Rotator Angle (binned in 5 degree intervals)")
plt.ylabel("Torque 1 (%)")
plt.title("Violin Plot of Torque 1 vs. Rotator Angle (binned by 5 degrees)")
plt.grid(True, alpha=0.3)

plt.show()


## Box plot

x-axis: Represents the rotator angle bin, which is binned in 5-degree intervals.

y-axis: Represents Torque 1 values for each of the binned rotator angles.

The box plot shows:

The median (middle line inside the box).

The interquartile range (IQR) (the box itself, from the 25th percentile to the 75th percentile).

The whiskers extend to the max and min values within 1.5 times the IQR.

Any outliers are shown as points outside of the whiskers.

The results are still the same, except that we have a really good representation of outliers. Even in this case the torque doesn't pass the 50%

In [ ]:
# Bin the data by 5 degrees
day_obs_df['rotator_angle_bin'] = (day_obs_df['rotator_actual_position'] // 5) * 5

plt.figure(figsize=(12, 6))

sns.boxplot(
    x='rotator_angle_bin', 
    y='torque_mean_1', 
    data=day_obs_df, 
    color="red",  
)

plt.xlabel("Rotator Angle (binned in 5-degree intervals)")
plt.ylabel("Torque 1 (%)")
plt.title("Box Plot of Torque 1 vs. Rotator Angle (binned by 5 degrees)")
plt.grid(True, alpha=0.3)

plt.show()


## Torque for positive and negative rotator angles

This is the same plot but we ust want to split the positive and negative rotator angles. The results are the same

In [ ]:
negative_angles = day_obs_df[day_obs_df['rotator_actual_position'] < 0]
positive_angles = day_obs_df[day_obs_df['rotator_actual_position'] >= 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)  # Two subplots, shared y-axis

# Plot for negative rotator angles
sns.scatterplot(
    x=negative_angles['rotator_actual_position'],
    y=negative_angles['torque_mean_0'],
    color='blue',
    s=50,
    marker='o',
    label="Torque 0",
    ax=axes[0]
)

sns.scatterplot(
    x=negative_angles['rotator_actual_position'],
    y=negative_angles['torque_mean_1'],
    color='red',
    s=50,
    marker='o',
    label="Torque 1",
    ax=axes[0]
)

axes[0].set_title("Negative Rotator Angles")
axes[0].set_xlabel("Rotator Actual Position (degrees)")
axes[0].set_ylabel("Torque (%)")
axes[0].legend()

# Plot for positive rotator angles
sns.scatterplot(
    x=positive_angles['rotator_actual_position'],
    y=positive_angles['torque_mean_0'],
    color='blue',
    s=50,
    marker='o',
    label="Torque 0",
    ax=axes[1]
)

sns.scatterplot(
    x=positive_angles['rotator_actual_position'],
    y=positive_angles['torque_mean_1'],
    color='red',
    s=50,
    marker='o',
    label="Torque 1",
    ax=axes[1]
)

axes[1].set_title("Positive Rotator Angles")
axes[1].set_xlabel("Rotator Actual Position (degrees)")
axes[1].legend()

plt.tight_layout()
plt.show()


## Torque and rotator angles for specific slews

Now we will analyse the torque VS rotator angle for different slews, represented by sequence numeber. The torque is represented with red (torque_1) and blue (torque_0). The rotator angle is represented with green line, and we can cleary see that there is rotator oscilation in between sequence numbers 250 and 320, whitch is affecting the torque but that is the only difference. We'll add oscilation analyses separately.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Torque Mean vs Sequence ID
sns.lineplot(
    x=day_obs_df["seq_num"],
    y=day_obs_df["torque_mean_0"],
    label="Torque 0 (Mean)",
    color="blue",
    ax=ax1
)
sns.lineplot(
    x=day_obs_df["seq_num"],
    y=day_obs_df["torque_mean_1"],
    label="Torque 1 (Mean)",
    color="red",
    ax=ax1
)

ax1.set_xlabel("Sequence ID (seq_num)")
ax1.set_ylabel("Torque Mean (%)", color="black")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
sns.lineplot(
    x=day_obs_df["seq_num"],
    y=day_obs_df["rotator_actual_position"],
    label="Rotator Position",
    color="green",
#    linestyle="dashed",
    ax=ax2
)
ax2.set_ylabel("Rotator Actual Position (degrees)", color="green")

plt.title("Torque Mean & Rotator Position vs. Sequence ID")
plt.show()


## Histogram of Torque by Rotator Angle

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    x=day_obs_df["rotator_actual_position"], 
    weights=day_obs_df["torque_mean_0"].abs(), 
    bins=30, 
    kde=True, 
    color="blue",
    label="Torque 0"
)
sns.histplot(
    x=day_obs_df["rotator_actual_position"], 
    weights=day_obs_df["torque_mean_1"].abs(), 
    bins=30, 
    kde=True, 
    color="red",
    label="Torque 1"
)

plt.xlabel("Rotator Actual Position (degrees)")
plt.ylabel("Sum of Absolute Torque (Weighted)")
plt.legend()
plt.title("Distribution of Torque Across Rotator Angles")
plt.grid(True, alpha=0.3)
plt.show()


## Heatmap (2D Histogram)

The heatmap bellow show that the most frequent torque for this day was around 20%. 

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.histplot(
    x=day_obs_df["rotator_actual_position"], 
    y=day_obs_df["torque_mean_1"].abs(), 
    bins=30, 
    cmap="magma",
    cbar=True
)

ax.set_xlabel("Rotator Actual Position (degrees)")
ax.set_ylabel("Torque Mean 1 (Absolute)")
ax.set_title("Torque vs. Rotator Angle Heatmap")

ax2 = ax.twinx()
ax2.set_ylabel("Torque Mean 1 (colourmap)")  

cbar = ax.collections[-1].colorbar
cbar.ax.set_position([0.85, 0.15, 0.03, 0.7])

plt.grid(True, alpha=0.3)

plt.show()



## The whole campaign

In [ ]:
from astropy.time import Time
from datetime import datetime, timedelta 

In [ ]:
day_obs_start = 20241024
day_obs_end = 20241211
fmt = "%Y%m%d"

t_start = Time(datetime.strptime(str(day_obs_start), fmt))
t_end = Time(datetime.strptime(str(day_obs_end), fmt))

current_date = t_start
df_list = []

while current_date <= t_end:
    next_date = current_date + timedelta(days=7)  # Query one week at a time

    query = f"""
    SELECT 
        min(actualTorquePercentage0) as min_torque_0,
        mean(actualTorquePercentage0) as mean_torque_0,
        max(actualTorquePercentage0) as max_torque_0,
        min(actualTorquePercentage1) as min_torque_1,
        mean(actualTorquePercentage1) as mean_torque_1,
        max(actualTorquePercentage1) as max_torque_1,
        mean(actualPositionTimestamp) as avg_timestamp  -- Using mean to get a representative timestamp
    FROM "lsst.sal.MTMount.cameraCableWrap"
    WHERE time >= '{t_start.isot}Z'
    AND time <= '{t_end.isot}Z'
    GROUP BY time(24h)
    """

    # Run the query
    temp_df = await efd_client.influx_client.query(query)

    if not temp_df.empty:
        df_list.append(temp_df)

    # Move to the next batch
    current_date = next_date

# Merge all partial dataframes
if df_list:
    campaign_df = pd.concat(df_list, ignore_index=True)
    print("Final dataframe shape:", campaign_df.shape)
else:
    campaign_df = pd.DataFrame()
    print("No data found!")



In [ ]:
campaign_df

In [ ]:
campaign_start = 20241024
campaign_end = 20241211

campaign_df = pd.DataFrame(
    columns=[
        "day_obs",
        "seq_num",
        "torque_min_0",
        "torque_avg_0",
        "torque_max_0",
        "torque_min_1",
        "torque_avg_1",
        "torque_max_1",
        "max_abs_torque_0",
        "max_abs_torque_1",
        "rotator_actual_position",
    ]
)

for day_obs in range(campaign_start, campaign_end + 1):
    all_events = event_maker.getEvents(day_obs)
    slew_events = [e for e in all_events if e.type == TMAState.SLEWING]

    for evt in slew_events:
        df = getEfdData(
            client=efd_client,
            topic="lsst.sal.MTMount.cameraCableWrap",
            columns=[
                "actualPosition",
                "actualTorquePercentage0",
                "actualTorquePercentage1",
                "actualVelocity",
            ],
            event=evt,
            warn=False,
        )

        if len(df) == 0:
            print(f"dayObs = {evt.dayObs}, seqNum = {evt.seqNum} - Empty dataframe")
            continue

        # Fetch rotator position
        rot = getEfdData(
            client=efd_client,
            topic="lsst.sal.MTPtg.mountPosition",
            columns=["rotatorActualPosition"],
            event=evt,
        )

        
        rot_value = rot["rotatorActualPosition"].iloc[0] if not rot.empty else None

        
        stats = {
            f"torque_{stat}_{i}": df[f"actualTorquePercentage{i}"].agg(stat)
            for i in [0, 1]
            for stat in ["min", "mean", "max"]
        }
        stats["seq_num"] = evt.seqNum
        stats["day_obs"] = day_obs
        stats["max_abs_torque_0"] = df["actualTorquePercentage0"].abs().max()
        stats["max_abs_torque_1"] = df["actualTorquePercentage1"].abs().max()
        stats["rotator_actual_position"] = rot_value

        
        campaign_df = pd.concat([campaign_df, pd.DataFrame([stats])], ignore_index=True)


In [ ]:
campaign_df

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot for Mean Torque 0 and Torque 1
sns.scatterplot(
    x=campaign_df["rotator_actual_position"], 
    y=campaign_df["torque_avg_0"],  
    label="Torque 0 (Mean)", 
    color="blue",
    alpha=0.7,  # Transparency to reduce overlap
    s=10  # Size of points
)

sns.scatterplot(
    x=campaign_df["rotator_actual_position"], 
    y=campaign_df["torque_avg_1"],  
    label="Torque 1 (Mean)", 
    color="red",
    alpha=0.7,
    s=10
)

# Scatter plot for Min and Max Torque 0
sns.scatterplot(
    x=campaign_df["rotator_actual_position"],
    y=campaign_df["torque_min_0"],
    color="blue",
    alpha=0.3,
    s=5,
    label="Torque 0 (Min)"
)

sns.scatterplot(
    x=campaign_df["rotator_actual_position"],
    y=campaign_df["torque_max_0"],
    color="blue",
    alpha=0.3,
    s=5,
    label="Torque 0 (Max)"
)

# Scatter plot for Min and Max Torque 1
sns.scatterplot(
    x=campaign_df["rotator_actual_position"],
    y=campaign_df["torque_min_1"],
    color="red",
    alpha=0.3,
    s=5,
    label="Torque 1 (Min)"
)

sns.scatterplot(
    x=campaign_df["rotator_actual_position"],
    y=campaign_df["torque_max_1"],
    color="red",
    alpha=0.3,
    s=5,
    label="Torque 1 (Max)"
)

plt.xlabel("Rotator Actual Position (degrees)")
plt.ylabel("Torque (%)")
plt.title("Torque vs. Rotator Position (Full Campaign Data)")
plt.legend()
plt.grid(True, alpha=0.3)

# Show plot
plt.show()


This plot shows the scatter of the CCW torques depending on rotator angle for The whole ComCam campaign.

The next plot will do the same just for the mean torque values. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.lines as mlines

plt.figure(figsize=(10, 6))

# Scatter plot for Torque 0 (Mean)
scatter_0 = sns.scatterplot(
    x=campaign_df["rotator_actual_position"], 
    y=campaign_df["torque_mean_0"],  
    color="blue", 
    s=50  
)

# Scatter plot for Torque 1 (Mean)
scatter_1 = sns.scatterplot(
    x=campaign_df["rotator_actual_position"], 
    y=campaign_df["torque_mean_1"],  
    color="red", 
    s=50  
)

legend_torque_0 = mlines.Line2D([], [], color="blue", marker="o", linestyle="None", markersize=6, label="Torque 0 (Mean)")
legend_torque_1 = mlines.Line2D([], [], color="red", marker="o", linestyle="None", markersize=6, label="Torque 1 (Mean)")

plt.xlabel("Rotator Actual Position (degrees)")
plt.ylabel("Torque (%)")
plt.title("Torque vs. Rotator Position (Full Campaign Data)")
plt.legend(handles=[legend_torque_0, legend_torque_1], title="Legend")  # Explicitly define the legend
plt.grid(True, alpha=0.3)

plt.show()


## Violin plot for the whole campaign

1. A wide portion of the violin at, say, -5° would mean that torque values are frequently around that angle.
   
2. A narrower portion at 20° suggests that torque values at that angle are less frequent or less consistent.

3. The median line within the violin gives a good sense of the central tendency (where most of the data are centered).

4. and, if there are any thin tails (i.e., the violin is narrow at some values), it suggests that data points at those values are sparse.

Again, the results are not showing any significant dependence between the torques and the rotator angles. 

In [ ]:
# Bin the data by 5 degrees
campaign_df['rotator_angle_bin'] = (campaign_df['rotator_actual_position'] // 5) * 5

plt.figure(figsize=(12, 6))

# Create the violin plot
sns.violinplot(
    x='rotator_angle_bin', 
    y='torque_mean_1', 
    data=campaign_df, 
    inner="points", 
    color="red",     
)

plt.xlabel("Rotator Angle (binned in 5 degree intervals)")
plt.ylabel("Torque 1 (%)")
plt.title("Violin Plot of Torque 1 vs. Rotator Angle (binned by 5 degrees)")
plt.grid(True, alpha=0.3)

plt.show()


## Box plot

x-axis: Represents the rotator angle bin, which is binned in 5-degree intervals.

y-axis: Represents Torque 1 values for each of the binned rotator angles.

The box plot shows:

The median (middle line inside the box).

The interquartile range (IQR) (the box itself, from the 25th percentile to the 75th percentile).

The whiskers extend to the max and min values within 1.5 times the IQR.

Any outliers are shown as points outside of the whiskers.

The results are still the same, except that we have a really good representation of outliers. Even in this case, the torque doesn't pass the 50% for the whole ComCam campaign. 

In [ ]:
# Bin the data by 5 degrees
campaign_df['rotator_angle_bin'] = (campaign_df['rotator_actual_position'] // 5) * 5

plt.figure(figsize=(12, 6))

sns.boxplot(
    x='rotator_angle_bin', 
    y='torque_mean_1', 
    data=campaign_df, 
    color="red",  
)

plt.xlabel("Rotator Angle (binned in 5-degree intervals)")
plt.ylabel("Torque 1 (%)")
plt.title("Box Plot of Torque 1 vs. Rotator Angle (binned by 5 degrees)")
plt.grid(True, alpha=0.3)

plt.show()

Now we'll do the torque VS rotator angle for the specific slews, for the whole campaign. The results show no significant dependence. 

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

sns.lineplot(
    x=campaign_df["seq_num"],
    y=campaign_df["torque_mean_0"],
    label="Torque 0 (Mean)",
    color="blue",
    ax=ax1
)
sns.lineplot(
    x=campaign_df["seq_num"],
    y=campaign_df["torque_mean_1"],
    label="Torque 1 (Mean)",
    color="red",
    ax=ax1
)

ax1.set_xlabel("Sequence ID (seq_num)")
ax1.set_ylabel("Torque Mean (%)", color="black")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)

# Create a second y-axis for rotator position
ax2 = ax1.twinx()
sns.lineplot(
    x=campaign_df["seq_num"],
    y=campaign_df["rotator_actual_position"],
    label="Rotator Position",
    color="green",
#    linestyle="dashed",
    ax=ax2
)
ax2.set_ylabel("Rotator Actual Position (degrees)", color="green")

plt.title("Torque Mean & Rotator Position vs. Sequence ID")
plt.show()


## Heatmap (2D Histogram) for the whole campaign

This plot shows that the most promimant torques are between 1% and 20% for the whole ComCam campaign. 

In [ ]:
plt.figure(figsize=(10, 6))

ax = sns.histplot(
    x=campaign_df["rotator_actual_position"], 
    y=campaign_df["torque_mean_1"].abs(), 
    bins=30, 
    cmap="magma",
    cbar=True
)

ax.set_xlabel("Rotator Actual Position (degrees)")
ax.set_ylabel("Torque Mean 1 (Absolute)")
ax.set_title("Torque vs. Rotator Angle Heatmap")

ax2 = ax.twinx()
ax2.set_ylabel("Torque Mean 1 (number of events)")  

cbar = ax.collections[-1].colorbar
cbar.ax.set_position([0.85, 0.15, 0.03, 0.7])

plt.grid(True, alpha=0.3)

plt.show()